# Fine-Tuning with LoRA and QLoRA

first let's get the right libraries

In [1]:
# %pip install -U peft transformers

Then get a base model

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side='left',
    clean_up_tokenization_spaces=True
)

In [3]:
tokenizer.special_tokens_map

{'bos_token': '</s>',
 'eos_token': '</s>',
 'unk_token': '</s>',
 'pad_token': '<pad>'}

Now let's get some data for the example. One dataset in spanish and one dataset in french. We are not going to fully train a model but we are going to look at the logic if actually wanted to do it

In [4]:
from datasets import load_dataset

spanish_data = load_dataset('andreamorgar/spanish_poetry')
french_data = load_dataset('Abirate/french_book_reviews')

In [5]:
spanish_data['train']['content'][3]

'\n\nLos dedos de la nieve\nrepiquetearon\nen el tamboril\ndel espacio.\n\nParábolas de nubes\nforman un halo\nde cristal,\nsobre el monte nevado.\n\nUna línea\ny un plano.\n\nQuiero poner mi vista\nsólo en el espacio,\nque es sencillo\ny a la vez complicado.'

In [6]:
french_data['train']['reader_review'][2]

"Pour écrire La plus secrète mémoire des hommes, Mohamed Mbougar Sarr s'est inspiré du destin brisé de Yambo Ouologuem, premier écrivain africain à remporter le Prix Renaudot en 1968 avec Le devoir de violence, à 28 ans. Il a connu la gloire, puis l’opprobre, finissant sa vie reclus, honni par ses pairs."

Now let's get that data ready for training by tokenizing it

In [7]:
max_length = 128

def preprocess_spanish(examples):
    return tokenizer(
        [x for x in examples['content'] if x], 
        max_length=max_length,
        truncation=True, 
        padding='max_length'
    )

def preprocess_french(examples):
    return tokenizer(
        [x for x in examples['reader_review'] if x], 
        max_length=max_length,
        truncation=True, 
        padding='max_length'
    )

tokenized_spanish = spanish_data.map(
    preprocess_spanish,
    batched=True,
    remove_columns=spanish_data['train'].column_names,
)

tokenized_french = french_data.map(
    preprocess_french,
    batched=True,
    remove_columns=french_data['train'].column_names,
)

In [8]:
tokenized_french

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 9645
    })
})

Now let's some LoRA adapters. We start by setting the config

In [9]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=64,
    task_type="CAUSAL_LM",
    # target_modules={'q_proj', 'v_proj', 'embed_tokens'}
)

In [10]:
print(lora_config)

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type='CAUSAL_LM', inference_mode=False, r=64, target_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


In [11]:
model = AutoModelForCausalLM.from_pretrained(model_id)
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

And then, we add the adpater for fine-tuning the model for spanish causal language modeling

In [12]:
model.add_adapter(lora_config, adapter_name='spanish_adapter')

In [13]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
              (lora_dropout): ModuleDict(
                (spanish_adapter): Identity()
              )
              (lora_A): ModuleDict(
                (spanish_adapter): Linear(in_features=1024, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (spanish_adapter): Linear(in_features=64, out_features=1024, bia

In [14]:
model.add_adapter(lora_config, adapter_name='french_adapter')

In [15]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
              (lora_dropout): ModuleDict(
                (spanish_adapter): Identity()
                (french_adapter): Identity()
              )
              (lora_A): ModuleDict(
                (spanish_adapter): Linear(in_features=1024, out_features=64, bias=False)
                (french_adapter): Linear(in_features=1024, out_features=64, bias=False)

In [16]:
model.active_adapters()

['french_adapter']

In [17]:
model.set_adapter('spanish_adapter')

In [18]:
model.active_adapters()

['spanish_adapter']

There is another way to add apdaters. Let's get back the model

In [19]:
model = AutoModelForCausalLM.from_pretrained(model_id)
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

In [20]:
from peft import get_peft_model

peft_model = get_peft_model(
    model, 
    lora_config, 
    adapter_name='spanish_adapter'
)

In [21]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (spanish_adapter): Identity()
                  )
                  (lora_A): ModuleDict(
                    (spanish_adapter): Linear(in_features=1024, out_features=64, bias=False)
  

In [22]:
peft_model.get_base_model()

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
              (lora_dropout): ModuleDict(
                (spanish_adapter): Identity()
              )
              (lora_A): ModuleDict(
                (spanish_adapter): Linear(in_features=1024, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (spanish_adapter): Linear(in_features=64, out_features=1024, bia

In [23]:
peft_model.add_adapter(
    adapter_name='french_adapter', 
    peft_config=lora_config
)

peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (spanish_adapter): Identity()
                    (french_adapter): Identity()
                  )
                  (lora_A): ModuleDict(
                    (spanish_adapter): Linear(

Let's get the causal language data collator for training

In [24]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False
)

And let's train for spanish

In [25]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./result_training",
    learning_rate=2e-5,
    weight_decay=0.01,
)

peft_model.set_adapter('spanish_adapter')

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_spanish['train'],
    data_collator=data_collator,
)

trainer.train()

[2024-09-06 22:52:00,958] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: gulmert89 (gulmert89-kariyer-net). Use `wandb login --relogin` to force relogin


  0%|          | 0/1926 [00:00<?, ?it/s]

{'loss': 3.5396, 'grad_norm': 0.26218554377555847, 'learning_rate': 1.4807892004153688e-05, 'epoch': 0.78}
{'loss': 3.4349, 'grad_norm': 0.28292232751846313, 'learning_rate': 9.615784008307374e-06, 'epoch': 1.56}
{'loss': 3.4125, 'grad_norm': 0.31849732995033264, 'learning_rate': 4.42367601246106e-06, 'epoch': 2.34}
{'train_runtime': 410.0005, 'train_samples_per_second': 37.544, 'train_steps_per_second': 4.698, 'train_loss': 3.4537184290425427, 'epoch': 3.0}


TrainOutput(global_step=1926, training_loss=3.4537184290425427, metrics={'train_runtime': 410.0005, 'train_samples_per_second': 37.544, 'train_steps_per_second': 4.698, 'total_flos': 3734997288615936.0, 'train_loss': 3.4537184290425427, 'epoch': 3.0})

In [26]:
base_model = peft_model.get_base_model()

In [27]:
def generate_text(prompt, model):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(output[0]) 

base_model.to('cpu')
generate_text('Como estas?', base_model)

'</s>Como estas?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n\n¿Qué es la vida?\n'

In [28]:

peft_model.to('cpu')
peft_model.set_adapter('spanish_adapter')
generate_text('Como estas?', peft_model)

'</s>Como estas?\n\n¿Qué es el más grande?\n\n¿Qué es el más grande?\n\n¿Qué es el más grande?\n\n¿Qué es el más grande?\n\n¿Qué es el más grande?\n\n¿Qué es el más grande?\n\n¿Qué es el más grande?\n\n¿Qué es el m'

Now let's train for french

In [30]:
training_args = TrainingArguments(
    output_dir="./result_training",
    learning_rate=2e-5,
    weight_decay=0.01,
)

peft_model.to('cuda')
peft_model.set_adapter('french_adapter')

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_french['train'],
    data_collator=data_collator,
)

trainer.train()

  0%|          | 0/3618 [00:00<?, ?it/s]

{'loss': 3.381, 'grad_norm': 6.098206996917725, 'learning_rate': 1.7236042012161415e-05, 'epoch': 0.41}
{'loss': 3.5305, 'grad_norm': 133.63323974609375, 'learning_rate': 1.4472084024322832e-05, 'epoch': 0.83}
{'loss': 3.6927, 'grad_norm': 22.119396209716797, 'learning_rate': 1.1708126036484247e-05, 'epoch': 1.24}
{'loss': 3.7616, 'grad_norm': 15.470325469970703, 'learning_rate': 8.944168048645662e-06, 'epoch': 1.66}
{'loss': 3.7829, 'grad_norm': 663.4329833984375, 'learning_rate': 6.180210060807076e-06, 'epoch': 2.07}
{'loss': 3.8325, 'grad_norm': 7668.30615234375, 'learning_rate': 3.416252072968491e-06, 'epoch': 2.49}
{'loss': 3.8639, 'grad_norm': 5968.470703125, 'learning_rate': 6.522940851299061e-07, 'epoch': 2.9}
{'train_runtime': 765.1305, 'train_samples_per_second': 37.817, 'train_steps_per_second': 4.729, 'train_loss': 3.6978417054403545, 'epoch': 3.0}


TrainOutput(global_step=3618, training_loss=3.6978417054403545, metrics={'train_runtime': 765.1305, 'train_samples_per_second': 37.817, 'train_steps_per_second': 4.729, 'total_flos': 7020863155077120.0, 'train_loss': 3.6978417054403545, 'epoch': 3.0})

In [33]:
base_model
print(generate_text('Comment ca va?', base_model))

</s>Comment ca va? Je suis un peu dans de la même. Je suis en train de l'autre vrai vrai direment en train de direter. Jeunes femme qui a fait de la même, et qui a fait un peu de l'autre, et qui a fait un peu de l'autre, et qui a été une mémoire de la même de l'aut


In [34]:
peft_model.set_adapter('french_adapter')
print(generate_text('Comment ca va?', peft_model))

</s>Comment ca va? Je suis un peu de la méditeurie de la vie de la mère, et je suis un peu de la mère, et je suis un peu de mémoire, et je suis un peu de la mère. Je suis un peu de la mère, et je suis un peu de la mère. Je suis un peu de la mère, et je suis un pe


We can save the adapters

In [35]:
peft_model.save_pretrained('peft_adapters')

We can load them back on

In [36]:
from peft import PeftModelForCausalLM

model_spanish = PeftModelForCausalLM.from_pretrained(
    model,
    'peft_adapters/spanish_adapter'
)

We can merge the adpaters into a new one

In [37]:
peft_model.add_weighted_adapter(
    ['spanish_adapter', 'french_adapter'], 
    [0.5, 0.5], 
    adapter_name='new_adapter')

In [38]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (spanish_adapter): Identity()
                    (french_adapter): Identity()
                    (default): Identity()
                    (new_adapter): Identity()
                  

We can infer on multiple adapters at once -> Multi-LoRA

In [39]:
inputs = tokenizer(
    [
        "Hello",
        "Bonjour",
        "Hola",
    ],
    return_tensors="pt",
    padding=True,
)

adapter_names = [
    "__base__", 
    "french_adapter",
    "spanish_adapter",
]

peft_model.eval()

output = peft_model.generate(
    **inputs, 
    adapter_names=adapter_names, 
    max_new_tokens=20
)

In [40]:
tokenizer.decode(output[0]) 

"<pad><pad></s>Hello, I'm a newbie to this sub. I'm looking for a good place to start."

In [41]:
tokenizer.decode(output[1]) 

'</s>Bonjour, je suis un peu de la vie, mais je ne suis pas un'

In [42]:
tokenizer.decode(output[2]) 

'<pad></s>Hola, mi amor es que no me gusta, pero no me gusta.\n\n'

Let's quantize our model

In [44]:
# %pip install -U bitsandbytes --no-cache-dir

Unfortunally, we need GPU to be able to quantize

In [49]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=config,
    low_cpu_mem_usage=False
)

model = prepare_model_for_kbit_training(model)

Once quantized, we can load a LoRa config

In [50]:
from peft import get_peft_model

peft_model = get_peft_model(
    model, 
    lora_config, 
    adapter_name='spanish_adapter'
)

In [51]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
          (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (spanish_adapter): Identity()
                  )
                  (lora_A): ModuleDict(
                    (spanish_adapter): Linear(in_features=1024, out_feature